- Miguel Baños Baladrón
- Fiz Garrido Escudero

# Práctica 1: Clasificación

### Importaciones

In [ ]:
import os
import glob
import torch
import pandas as pd
import torchvision.models as models
import torch.nn as nn
import torch.optim as optim

from torch.utils import data
from torch.utils.data import Subset
from torch.utils.data import DataLoader
from torchvision.io import read_image
from torchvision import transforms
from sklearn.model_selection import train_test_split

# 1.- Creación de Dataset personalizado

In [ ]:
class ShipDataset(data.Dataset):
    def __init__(self, image_path, ship_csv, docked_csv, transform=None):
        super().__init__()
        # Guardamos la ruta de cada imagen
        self.image_files = glob.glob(os.path.join(image_path, '*.jpg')) + \
                           glob.glob(os.path.join(image_path, '*.png'))

        # Leer csv
        ship = pd.read_csv(ship_csv, header=None)
        docked = pd.read_csv(docked_csv, header=None)

        # Diccionario de etiquetas: filename -> [ship_label, docked_label]
        self.labels = {}
        for i in range(len(ship)):
            filename = ship.iloc[i, 0]
            ship_label = ship.iloc[i, 1]
            docked_label = docked.iloc[i, 1]
            self.labels[filename] = [ship_label, docked_label]

        # Transformaciones: ToPILImage es necesario porque read_image devuelve un tensor
        if transform:
            self.transform = transform
        else:
            self.transform = transforms.Compose([
                transforms.ToPILImage(),
                transforms.ToTensor()
            ])

    def __getitem__(self, index):
        image_path = self.image_files[index]
        image_name = os.path.splitext(os.path.basename(image_path))[0]
        image = read_image(image_path)             # Tensor [C, H, W], uint8
        image = self.transform(image)              # Aplicamos transformaciones
        filename = os.path.basename(image_path)
        labels = torch.tensor(self.labels[filename], dtype=torch.float32)
        return image, labels, image_name

    def __len__(self):
        return len(self.image_files)

# 2.- Clasificación Ship/No-Ship

In [ ]:
IMAGE_PATH  = r"C:\Users\Pc\Desktop\Uni\Tercero\segundo_cuatri\VCA\VCA\P1-Material\images"
SHIP_CSV    = r"C:\Users\Pc\Desktop\Uni\Tercero\segundo_cuatri\VCA\VCA\P1-Material\ship.csv"
DOCKED_CSV  = r"C:\Users\Pc\Desktop\Uni\Tercero\segundo_cuatri\VCA\VCA\P1-Material\docked.csv"

### Transforms

In [ ]:
# Transform base (sin augmentation) — para validación, test, y baseline de entrenamiento
# ToPILImage va primero porque read_image devuelve un tensor uint8
base_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Transform con Data Augmentation — solo para entrenamiento
aug_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

### División del dataset

El dataset se divide en tres subconjuntos: **entrenamiento (70%)**, **validación (15%)** y **test (15%)**.  
La división es **estratificada** sobre la etiqueta *ship/no-ship* para mantener la proporción de clases en los tres conjuntos.

**Nota importante**: calculamos los índices sobre el dataset sin transforms, y luego creamos datasets separados con el transform adecuado para cada conjunto. Así el conjunto de entrenamiento puede usar augmentation sin afectar a validación y test.

In [ ]:
# Dataset auxiliar (sin transform) solo para obtener etiquetas y calcular índices
dataset_aux = ShipDataset(IMAGE_PATH, SHIP_CSV, DOCKED_CSV)

filenames = dataset_aux.image_files
ship_labels = [dataset_aux.labels[os.path.basename(f)][0] for f in filenames]

In [ ]:
# Primer split: 70% train / 30% resto
train_files, temp_files, train_labels, temp_labels = train_test_split(
    filenames,
    ship_labels,
    test_size=0.30,
    stratify=ship_labels,
    random_state=42
)

# Segundo split: resto -> 50% val / 50% test  (= 15% / 15% del total)
val_files, test_files, val_labels, test_labels = train_test_split(
    temp_files,
    temp_labels,
    test_size=0.50,
    stratify=temp_labels,
    random_state=42
)

print(f"Train: {len(train_files)} | Val: {len(val_files)} | Test: {len(test_files)}")

In [ ]:
# Convertimos rutas a índices sobre filenames
file_to_idx = {f: i for i, f in enumerate(filenames)}

train_idx = [file_to_idx[f] for f in train_files]
val_idx   = [file_to_idx[f] for f in val_files]
test_idx  = [file_to_idx[f] for f in test_files]

In [ ]:
# Datasets con sus transforms correspondientes
# - train_dataset_base: sin augmentation (para el experimento baseline)
# - train_dataset_aug:  con augmentation
# - val/test siempre con base_transform

dataset_base = ShipDataset(IMAGE_PATH, SHIP_CSV, DOCKED_CSV, transform=base_transform)
dataset_aug  = ShipDataset(IMAGE_PATH, SHIP_CSV, DOCKED_CSV, transform=aug_transform)

train_dataset_base = Subset(dataset_base, train_idx)
train_dataset_aug  = Subset(dataset_aug,  train_idx)
val_dataset        = Subset(dataset_base, val_idx)
test_dataset       = Subset(dataset_base, test_idx)

### DataLoaders

In [ ]:
BATCH_SIZE = 32

def make_loaders(train_dataset):
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
    val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    return train_loader, val_loader, test_loader

# 3.- Modelo, entrenamiento y evaluación

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando: {device}")

def build_model():
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    model.fc = nn.Linear(model.fc.in_features, 1)  # Clasificación binaria
    return model.to(device)

In [ ]:
def train(model, train_loader, val_loader, epochs=10):
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4)

    for epoch in range(epochs):
        # --- Entrenamiento ---
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels, _ in train_loader:
            images = images.to(device)
            labels = labels[:, 0].unsqueeze(1).to(device)  # Etiqueta ship

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            preds = torch.sigmoid(outputs) > 0.5
            correct += (preds == labels.bool()).sum().item()
            total += labels.size(0)

        train_acc = correct / total

        # --- Validación ---
        model.eval()
        val_correct = 0
        val_total = 0
        val_loss = 0.0

        with torch.no_grad():
            for images, labels, _ in val_loader:
                images = images.to(device)
                labels = labels[:, 0].unsqueeze(1).to(device)
                outputs = model(images)
                val_loss += criterion(outputs, labels).item()
                preds = torch.sigmoid(outputs) > 0.5
                val_correct += (preds == labels.bool()).sum().item()
                val_total += labels.size(0)

        val_acc = val_correct / val_total
        print(f"Epoch {epoch+1:02d}/{epochs} | "
              f"Train Loss: {running_loss/len(train_loader):.4f} | Train Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss/len(val_loader):.4f} | Val Acc: {val_acc:.4f}")

    return model

In [ ]:
def evaluate(model, test_loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels, _ in test_loader:
            images = images.to(device)
            labels = labels[:, 0].unsqueeze(1).to(device)
            outputs = model(images)
            preds = torch.sigmoid(outputs) > 0.5
            correct += (preds == labels.bool()).sum().item()
            total += labels.size(0)

    acc = correct / total
    print(f"Test Accuracy: {acc:.4f}")
    return acc

## Experimento A: Sin Data Augmentation (baseline)

In [ ]:
print("=== BASELINE (sin augmentation) ===")
train_loader_base, val_loader, test_loader = make_loaders(train_dataset_base)
model_base = build_model()
model_base = train(model_base, train_loader_base, val_loader, epochs=10)
acc_base = evaluate(model_base, test_loader)

## Experimento B: Con Data Augmentation

In [ ]:
print("=== CON DATA AUGMENTATION ===")
train_loader_aug, val_loader, test_loader = make_loaders(train_dataset_aug)
model_aug = build_model()
model_aug = train(model_aug, train_loader_aug, val_loader, epochs=10)
acc_aug = evaluate(model_aug, test_loader)

## Comparación de resultados

In [ ]:
print(f"Baseline (sin aug):  {acc_base:.4f}")
print(f"Con augmentation:   {acc_aug:.4f}")
print(f"Diferencia:          {acc_aug - acc_base:+.4f}")